# 문항 2: in 연산자 기반 타입 가드 분기 처리 및 제네릭 인메모리 저장소(Storage) 구축

In [11]:
// 데이터 모델 (01_data_modeling.ipynb와 동일)

type UserRole = "admin" | "customer" | "guest";
type Category = "electronics" | "clothing" | "books";

interface User {
  id: number;
  name: string;
  email: string;
  role: UserRole;
  isActive: boolean;
}

interface Product {
  id: number;
  title: string;
  price: number;
  category: Category;
  stock: number;
  discountRate?: number; // 할인율 (0~1 비율)
}

// 테스트용 샘플 데이터
const sampleUser: User = { id: 1, name: "고객", email: "customer@shop.com", role: "customer", isActive: true };

const sampleProduct: Product = { id: 1, title: "컴퓨터", price: 10000, category: "electronics", stock: 40 };

## [필수]
- in 연산자 기반 타입 좁히기(Narrowing)를 활용한 유니언 데이터 분기 처리

In [12]:
// 타입 가드 함수 작성 (in 연산자를 활용한 타입 좁히기)

function renderItemSummary(item: User | Product) : string {
    
    if ("email" in item) {
        // email은 User에만 있는 프로퍼티
        return `[회원] ${item.name} (${item.role})` ;

    } else if ("title" in item) {
        // title은 Product에ㅔ만 있는 프로퍼티
        return `[상품] ${item.title} - ${item.price}원` ; 
    
    } else {
        // 모든 경우를 처리했다면 item은 never 타입
        // 나중에 유니언에 새 타입이 추가되면 이 줄에서 컴파일 오류가 나서 처리 누락을 알려 줌  
        const exhaustiveCheck: never = item ; 
        return exhaustiveCheck ;
    }
}

In [13]:
// 안전한 데이터 포맷팅 및 출력

const items: (User | Product)[] = [sampleUser, sampleProduct] ;


for (const item of items) {
    console.log(renderItemSummary(item)) ;
}

[회원] 고객 (customer)
[상품] 컴퓨터 - 10000원


## [심화]
- 제네릭 제약 조건(extends)을 적용한 범용 InMemoryStorage<T> 클래스 설계

In [14]:
// 제약 인터페이스 선언

// 저장소에 들어가는 모든 항목이 고유키(id)를 갖도록 보장
interface Identifiable {
    id : number ; 
}

In [15]:
// 제네릭 클래스 구현 및  CRUD 메서드 작성

// (제너릭 제약 조건) T는 반드시 id: number를 가진 타입이어야 함
class InMemoryStorage<T extends Identifiable> {
    // private: 클래스 외부에서 items에 직접 접근·수정할 수 없음 (캡슐화)
    private items : T[] = [];

    // [C 항목 추가] - 같은 id가 이미 있으며 추가하지 않음
    add(item : T) : void {
        if (this.getById(item.id)) {
            console.warn(`id ${item.id}는 이미 존재하여 추가하지 않습니다`)
            return ; 
        }
        this.items.push(item) ; // 없으면 추가
    }

    // [R 항목 조회] - 없으면 undefined
    getById(id : number) : T | undefined {
        return this.items.find((item) => item.id === id) ;
    }

    // 전체 목록을 복사해서 반환 - 외부에서 원본 배열을 수정하지 못하도록
    getAll() : T[] {
        return [...this.items] ;
    }
}

In [16]:
// 다형성 인스턴스 검증

// 1) User 저장소 
const userStorage = new InMemoryStorage<User>() ;
userStorage.add({ id: 1, name: "관리자", email: "admin@shop.com", role: "admin", isActive: true });
userStorage.add({ id: 2, name: "고객", email: "customer@shop.com", role: "customer", isActive: true });
userStorage.add({ id: 2, name: "중복", email: "dup@shop.com", role: "guest", isActive: false }); // 중복 id

console.log("전체 회원");
console.table(userStorage.getAll());

id 2는 이미 존재하여 추가하지 않습니다


전체 회원
┌─────────┬────┬──────────┬─────────────────────┬────────────┬──────────┐
│ (index) │ id │ name     │ email               │ role       │ isActive │
├─────────┼────┼──────────┼─────────────────────┼────────────┼──────────┤
│ 0       │ 1  │ '관리자' │ 'admin@shop.com'    │ 'admin'    │ true     │
│ 1       │ 2  │ '고객'   │ 'customer@shop.com' │ 'customer' │ true     │
└─────────┴────┴──────────┴─────────────────────┴────────────┴──────────┘


In [17]:
const foundUser = userStorage.getById(2); // 타입: User | undefined
console.log("id 2 회원:", foundUser?.name);
console.log("id 99 회원:", userStorage.getById(99)); // 없는 id → undefined

id 2 회원: 고객
id 99 회원: undefined


In [18]:
// 2) Product 저장소
const productStorage = new InMemoryStorage<Product>();
productStorage.add({ id: 1, title: "컴퓨터", price: 10000, category: "electronics", stock: 40, discountRate: 0.2 });
productStorage.add({ id: 2, title: "티셔츠", price: 20000, category: "clothing", stock: 100 });

console.log("전체 상품");
console.table(productStorage.getAll());

const foundProduct = productStorage.getById(1); // 타입: Product | undefined
console.log("id 1 상품:", foundProduct?.title);

전체 상품
┌─────────┬────┬──────────┬───────┬───────────────┬───────┬──────────────┐
│ (index) │ id │ title    │ price │ category      │ stock │ discountRate │
├─────────┼────┼──────────┼───────┼───────────────┼───────┼──────────────┤
│ 0       │ 1  │ '컴퓨터' │ 10000 │ 'electronics' │ 40    │ 0.2          │
│ 1       │ 2  │ '티셔츠' │ 20000 │ 'clothing'    │ 100   │              │
└─────────┴────┴──────────┴───────┴───────────────┴───────┴──────────────┘
id 1 상품: 컴퓨터


## 구현 정리
- **타입 가드**: `'email' in item`으로 User/Product를 구분하고, 각 블록에서 해당 타입의 프로퍼티만 안전하게 사용
- **Exhaustive 검사**: 마지막 `else`에서 `never` 타입을 사용해, 유니언에 새 타입이 추가되면 처리 누락을 컴파일 오류로 알 수 있음
- **제네릭 제약**: `T extends Identifiable`로 모든 항목이 `id: number`를 갖도록 보장 → `getById`에서 `item.id`를 안전하게 사용
- **캡슐화**: `private items`로 외부 직접 접근을 막고, `getAll()`은 복사본을 반환해 원본 데이터를 보호
- **재사용성**: 하나의 클래스로 `User`, `Product` 저장소를 모두 생성하며, `id`만 있으면 어떤 타입이든 확장 가능